# nb_02_silver: clean, conform, dedupe, filter, without losing live rows
**Task 2.** Show the damage first, then fix it deliberately. The one idea this notebook is really testing is the **null-safe filter**.

In [ ]:
from pyspark.sql import functions as F, Window

bc = spark.table("bronze_customers")
bd = spark.table("bronze_depots")
br = spark.table("bronze_rentals")
bb = spark.table("bronze_billing")

def save(df, name):
    (df.write.mode("overwrite").format("delta")
       .option("overwriteSchema", "true").saveAsTable(name))
    print(f"{name:30s} rows = {spark.table(name).count()}")

## 1. Show the damage first

In [ ]:
print("CUSTOMERS  rows:", bc.count(), "| distinct CUSTOMER_ID:", bc.select("CUSTOMER_ID").distinct().count())
print("RENTALS    rows:", br.count(), "| distinct rental_id  :", br.select("rental_id").distinct().count())
print("BILLING    rows:", bb.count(), "| distinct BILL_ID    :", bb.select("BILL_ID").distinct().count())
print()
print("Distinct RAW spellings -> CUSTOMER_TYPE:", bc.select("CUSTOMER_TYPE").distinct().count(),
      "| RENTAL_TYPE:", br.select("rental_type").distinct().count(),
      "| PAYER_TYPE:", bb.select("PAYER_TYPE").distinct().count())
print("Spellings of CUSTOMER_TYPE after trim+lower (still more than 3):")
display(bc.groupBy(F.lower(F.trim("CUSTOMER_TYPE")).alias("customer_type")).count().orderBy(F.desc("count")))

## 2. Dedupe helper: keep the latest ingested row per key
`current_timestamp()` has the same value for every row of one load, so ties are common. The tie-breakers (`source_file`, then row id) make the choice deterministic.

In [ ]:
def latest(df, key):
    df = df.withColumn("_rid", F.monotonically_increasing_id())
    w = (Window.partitionBy(key)
               .orderBy(F.col("ingested_at").desc(), F.col("source_file").desc(), F.col("_rid").desc()))
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn", "_rid")

## 3. silver_customers

In [ ]:
bc = bc.withColumn("CUSTOMER_ID", F.trim("CUSTOMER_ID"))
cust = latest(bc, "CUSTOMER_ID")
print("customers after dedupe:", cust.count())          # 600

t = F.lower(F.trim("CUSTOMER_TYPE"))
customer_type = (F.when(t.isin("individual", "ind"), "individual")
                  .when(t.isin("contractor", "contr."), "contractor")
                  .when(t.isin("company", "corporate", "corp"), "company"))

registered = F.coalesce(F.to_date(F.trim("REGISTERED_ON"), "dd-MM-yyyy"),
                        F.to_date(F.trim("REGISTERED_ON"), "yyyy/MM/dd"))

silver_customers = cust.select(
    F.col("CUSTOMER_ID").alias("customer_id"),
    F.initcap(F.regexp_replace(F.trim("CUSTOMER_NAME"), r"\s+", " ")).alias("customer_name"),
    registered.alias("registered_on"),
    F.to_date(F.trim("KYC_VERIFIED_ON"), "yyyy-MM-dd").alias("kyc_verified_on"),
    customer_type.alias("customer_type"),
    F.trim("CITY").alias("city"),
    "ingested_at", "source_file")

# Parse-failure assert + how many a single-format parse would have silently nulled
parse_failures = silver_customers.filter(F.col("registered_on").isNull()).count()
single_format_nulls = cust.filter(F.to_date(F.trim("REGISTERED_ON"), "dd-MM-yyyy").isNull()).count()
assert parse_failures == 0, f"{parse_failures} REGISTERED_ON values failed both formats"
print(f"Date parse failures with coalesce of both formats: {parse_failures}")
print(f"A single-format parse would have silently nulled {single_format_nulls} of {silver_customers.count()} dates")   # 273 of 600

assert silver_customers.filter(F.col("customer_type").isNull()).count() == 0
print("customer_type distinct values:", sorted(r[0] for r in silver_customers.select("customer_type").distinct().collect()))

## 4. silver_depots (already clean: just type it)

In [ ]:
silver_depots = bd.select(
    F.upper(F.trim("DEPOT_CODE")).alias("depot_code"),
    F.trim("DEPOT_NAME").alias("depot_name"),
    F.trim("ZONE").alias("zone"),
    F.col("FLEET_SIZE").cast("int").alias("fleet_size"),
    "ingested_at", "source_file")

## 5. silver_rentals: dedupe, standardise, parse, then the NULL-SAFE quality rule

In [ ]:
br = (br.withColumn("rental_id", F.trim("rental_id"))
        .withColumn("customer_id", F.trim("customer_id"))
        .withColumn("depot_code", F.upper(F.trim("depot_code")))
        .withColumn("asset_id", F.upper(F.trim("asset_id"))))

rent = latest(br, "rental_id")
print("rentals after dedupe (re-sent day removed):", rent.count())      # 945

rt = F.lower(F.trim("rental_type"))
rent = (rent
    .withColumn("rental_type", F.when(rt == "standard", "standard").when(rt == "priority", "priority"))
    .withColumn("checkout_ts", F.to_timestamp(F.trim("checkout_ts")))
    # blank check-in = machine still out. It must stay NULL, never be 'fixed'.
    .withColumn("checkin_ts", F.to_timestamp(F.when(F.trim("checkin_ts") != "", F.trim("checkin_ts")))))

assert rent.filter(F.col("rental_type").isNull()).count() == 0
assert rent.filter(F.col("checkout_ts").isNull()).count() == 0
print("rental_type distinct values:", sorted(r[0] for r in rent.select("rental_type").distinct().collect()))

### The null-safe filter (the one idea this assignment is testing)
A check-in before the check-out is impossible. But for a machine still out, `checkin_ts` is NULL, so `NULL < checkout_ts` is **NULL, not false**. Then `~bad` is also NULL, and `filter` drops NULL rows, silently deleting every machine still on site.

In [ ]:
bad = F.col("checkin_ts") < F.col("checkout_ts")

careless_ok = rent.filter(~bad)                                  # WRONG: drops still-out rows
rent_ok     = rent.filter(~F.coalesce(bad, F.lit(False)))        # RIGHT: NULL is treated as "not bad"
rent_bad    = rent.filter(F.coalesce(bad, F.lit(False)))         # quarantine

still_out = rent_ok.filter(F.col("checkin_ts").isNull()).count()
dropped_by_careless = rent_ok.count() - careless_ok.count()
print(f"silver_rentals rows            : {rent_ok.count()}")           # 942
print(f"quarantined (checkin < checkout): {rent_bad.count()}")          # 3
print(f"still-out rentals kept         : {still_out}")                  # 175
print(f"the careless filter(~bad) would have dropped {dropped_by_careless} still-out rentals")   # 175
assert dropped_by_careless == still_out == 175

## 6. silver_billing: parse money, standardise payer type, LEFT join to rentals

In [ ]:
# "Rs. 64,157.16" and "26,037.89": strip the "Rs." prefix FIRST (its dot would survive a plain [^0-9.] clean), then commas/spaces.
amount = (F.regexp_replace(F.regexp_replace(F.trim("AMOUNT_INR"), r"(?i)rs\.?", ""), r"[^0-9.]", "")
           .cast("decimal(12,2)"))
pt = F.lower(F.trim("PAYER_TYPE"))

bill = bb.select(
    F.trim("BILL_ID").alias("bill_id"),
    F.trim("RENTAL_ID").alias("rental_id"),
    F.to_date(F.trim("BILL_DATE"), "yyyy-MM-dd").alias("bill_date"),
    amount.alias("amount_inr"),
    F.when(pt.isin("direct", "contract", "corporate", "prepaid"), pt).alias("payer_type"),
    F.col("AMOUNT_INR").alias("amount_raw"),
    "ingested_at", "source_file")

amount_failures = bill.filter(F.col("amount_inr").isNull()).count()
date_failures   = bill.filter(F.col("bill_date").isNull()).count()
assert amount_failures == 0, f"{amount_failures} amounts failed to parse"
assert date_failures == 0, f"{date_failures} bill dates failed to parse"
assert bill.filter(F.col("payer_type").isNull()).count() == 0
print(f"amount parse failures: {amount_failures} | bill_date parse failures: {date_failures}")
print("payer_type distinct values:", sorted(r[0] for r in bill.select("payer_type").distinct().collect()))

# LEFT join (never inner) so unmatched bills are visible, not silently lost
keys = rent_ok.select(F.col("rental_id").alias("_rental_key"))
joined = bill.join(keys, bill.rental_id == keys._rental_key, "left")

silver_billing           = joined.filter(F.col("_rental_key").isNotNull()).drop("_rental_key", "amount_raw")
silver_billing_unmatched = joined.filter(F.col("_rental_key").isNull()).drop("_rental_key")

print("silver_billing rows           :", silver_billing.count())             # 779
print("silver_billing_unmatched rows :", silver_billing_unmatched.count())   # 8

## 7. Write the Silver tables

In [ ]:
save(silver_customers,  "silver_customers")
save(silver_depots,     "silver_depots")
save(rent_ok,           "silver_rentals")
save(rent_bad,          "silver_rentals_quarantine")
save(silver_billing,    "silver_billing")
save(silver_billing_unmatched, "silver_billing_unmatched")

## 8. Acceptance checks (screenshot this cell's output)

In [ ]:
checks = {
    "silver_customers": 600, "silver_rentals": 942, "silver_rentals_quarantine": 3,
    "silver_billing": 779, "silver_billing_unmatched": 8, "silver_depots": 6,
}
for t, n in checks.items():
    got = spark.table(t).count()
    assert got == n, f"{t}: expected {n}, got {got}"
    print(f"OK  {t:28s} {got}")
assert spark.table("silver_customers").select("customer_type").distinct().count() == 3
assert spark.table("silver_rentals").select("rental_type").distinct().count() == 2
assert spark.table("silver_rentals").filter(F.col("checkin_ts").isNull()).count() == 175

print("\n--- numbers for your write-up ---")
print(f"single-format date parse would have nulled {single_format_nulls} of 600 dates")
print(f"careless filter(~bad) would have dropped {dropped_by_careless} still-out rentals")